# Board Diagrams Walkthrough
A section-by-section breakdown of every diagram produced for the Toronto Crash Risk Prediction system.

## 1. Hurdle Model Architecture

This flowchart shows the **two-stage hurdle pipeline**. Stage 1 is a HistGradientBoosting **binary classifier** (depth=6, 300 iterations) that predicts P(crash) and is then **isotonically calibrated**; Stage 2 is a **Poisson regressor** trained only on positive windows to estimate E[count | crash]. The two outputs are multiplied into a single **lambda = P_cal x E[n|crash]** (clipped to [0, 50]), which is then bucketed into **Low / Medium / High** risk at the **p70 and p90** percentile thresholds.

![Hurdle Model Architecture](01_hurdle_model_architecture.png)

## 2. Confusion Matrix

A **3x3 risk-classification confusion matrix** on ~10K samples. The **Low** class dominates with **6,650** correct predictions and **95% recall**. **High** risk achieves **70% recall** (700 out of 1,000 actual high-risk windows), and **overall accuracy is ~87%**. Most misclassifications happen between adjacent classes (Low ↔ Medium, Medium ↔ High) rather than extreme misses.

![Confusion Matrix](02_confusion_matrix.png)

## 3. Correlation Heatmap

A **14-feature Pearson correlation** matrix (lower triangle). The strongest cluster is **lag / historical** — crashes_1d_ago ↔ rolling_mean_7d at **r = 0.72**. Weather features form a negative cluster (temperature ↔ snow_depth at **r = −0.68**). Cross-group correlations are low, confirming **minimal multicollinearity** between feature groups.

![Correlation Heatmap](03_correlation_heatmap.png)

## 4. Toronto Risk Heatmap

A **spatial risk overlay** on the Toronto map showing predicted crash lambda values. The **downtown core** and **Yonge St corridor** show the highest risk, with visible hotspots at **Queen/Spadina**, **Bloor/Yonge**, **North York**, and **Scarborough**. Major roads — **Hwy 401, DVP, and the Gardiner** — also show elevated corridor risk.

![Toronto Risk Heatmap](04_toronto_risk_heatmap.png)

## 5. Data Pipeline Overview

An end-to-end pipeline flowchart. **3 raw sources** (618K collisions, 18K KSI records, 65K road segments) plus **4 enrichment sources** (weather, TMC traffic, schools, transit) feed into a **spatial join** (BallTree, 20m buffer), then through **7 feature groups** into the hurdle model. Predicted lambda values are bucketed into risk labels and served via **3 API endpoints** — including a **Dijkstra safety-aware router** (cost = travel_time + beta * expected_crashes).

![Data Pipeline Overview](05_data_pipeline_overview.png)

## 6. Data Flow Diagram

A formal **DFD** with external entities, processes, and data stores. It traces **8 processes** (data loader → spatial join → panel builder → model train → inference → labelling → Flask API → Dijkstra router) across **6 data stores**. This shows how raw collision Excel records flow end-to-end into **GeoJSON API responses** consumed by the **iOS app** and developer dashboard.

![Data Flow Diagram](06_dataflow_diagram.png)

## 7. SHAP Feature Importance

A horizontal bar chart of **mean |SHAP| values** across all 15 features. The top 3 are all **historical / lag** features: hist_crash_per_yr (**0.142**), rolling_mean_7d (**0.118**), and crashes_1d_ago (**0.095**). Temporal indicators (hour_sin, is_weekend, month_sin) contribute the least at **< 0.022** each — confirming that the model leans heavily on a segment's crash history.

![SHAP Feature Importance](07_shap_summary.png)

## 8. Calibration Curve

A **reliability diagram** (top) with a prediction histogram (bottom) for the Stage 1 binary classifier. **Isotonic calibration** pulls the predicted probabilities much closer to the perfect-calibration diagonal, improving the **Brier score by ~42%**. The histogram reveals extreme **class imbalance**: **42,000** samples in the lowest probability bin versus just **90** in the highest.

![Calibration Curve](08_calibration_curve.png)

## 9. System Architecture

A **deployment-view** diagram showing 3 zones. The **iOS client** (SwiftUI / MapKit) communicates over HTTPS/JSON with a **Flask server** that houses the API layer, model runtime, Dijkstra routing engine, Shapely spatial index, and feature pipeline. **Storage** includes the model `.pkl` artifact, centreline GeoJSON (65K segments), collision data (618K Excel + 18K CSV), and a weather cache.

![System Architecture](09_system_architecture.png)

## 10a. Crash Count Distribution

A **log-scale histogram** of crash counts per time window. **88%** of windows have **zero crashes** (zero-inflated), and the tail (3+ crashes) accounts for only **~1.5%** (~1,500 windows). This extreme imbalance is precisely why the model uses a **two-stage hurdle architecture** — Stage 1 separates zeros from non-zeros before Stage 2 models the positive counts.

![Crash Count Distribution](10a_crash_count_distribution.png)

## 10b. Risk Label Split

A **pie chart** showing the final risk label distribution: **70% Low**, **20% Medium**, **10% High**. These splits come directly from the **p70 and p90 percentile thresholds** applied to predicted lambda values across all road segments.

![Risk Label Split](10b_risk_label_split.png)

## 10c. Hurdle Model Filtering

A **stacked bar chart** showing how samples flow through the hurdle stages. Stage 1 filters out **88%** of windows (**88K zeros**); only the **12K positive** windows are passed to Stage 2. Tail-weighted sample weights (**w = 1 + 2.0 * log1p(y)** for y >= 2) upweight the rare high-count events so the model doesn't underfit the dangerous tail.

![Hurdle Model Filtering](10c_hurdle_filtering.png)